# DACIS — Notebook 3 (fixed): Dual-Gate Streaming Inference

Rewritten to fix six issues found in the previous version:

1. **No real time ordering existed** — nodes were processed in raw graph-construction index order, not chronological order. Now sorted by `data.time_step` (requires the patched Notebook 1 that attaches it — this notebook fails loudly if it's missing rather than silently falling back).
2. **Neighbor direction was backwards** — `to_scipy_sparse_matrix` puts source as the row, but `SAGEConv`'s default message flow aggregates a node's embedding from its *in*-neighbors. Every neighbor lookup here now uses the transposed (in-neighbor) adjacency, verified against the model's actual receptive field.
3. **Hop count didn't match the model** — the old code used raw 1-hop adjacency while scoring a 2-layer GraphSAGE model (2-hop receptive field), and a `L_HOPS` config existed but was never wired in. Now a real L-hop BFS is used, `L=2` by default, matching the model depth.
4. **Test-set threshold leakage** — F1 was previously computed by optimizing the decision threshold directly on the test labels being scored. Now the threshold is tuned on validation predictions and applied to test, matching Notebook 1/2's methodology.
5. **Unjustified score penalty** — the old fusion rule multiplicatively decayed the baseline score by 10% whenever Gate 2 didn't confirm. That's now a no-op (baseline score is left unchanged unless Gate 2 actively confirms an anomaly).
6. **No gate-purity diagnostic** — the previous version reported *how often* the gates fired but never *how often they were right*. This version tracks true-positive/false-positive counts for both gates so you can tell whether they're adding signal or just noise.

7. **Gate 2 was AND-gated against Gate 1, and untested** — the old fusion rule required Gate 2 (a Structural Invariance Check) to confirm before anything scored, with no check that its signal was actually valid. A fraud-vs-legit distribution check (see Phase 4) showed SIC added no real separation — and inverted on Elliptic, rejecting every true positive the distance-anomaly fallback found. Gate 2 has been replaced with a community-level Welford co-activation check, validated the same way, and is now **tiered on top of Gate 1 instead of AND-gated**: it can only add a confidence tier and a small extra score boost, never suppress a Gate 1 catch. This is guaranteed, by construction, to never do worse than Gate 1 alone.

Also added: a structural sanity check (fraction of test nodes with zero in-neighbors) printed up front for every dataset — this is the diagnostic that would have caught the AMLSim/PaySim node-isolation bug immediately instead of via a suspicious 0-trigger result three cells later.

In [1]:
%%capture
!pip install torch-geometric numpy pandas pyarrow networkx matplotlib seaborn scikit-learn scipy tqdm --quiet

In [2]:
from __future__ import annotations

import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
import torch_geometric.utils
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score
from scipy.sparse.csgraph import connected_components

pd.set_option("display.max_columns", 50)
sns.set_style("whitegrid")
warnings.filterwarnings("ignore")


@dataclass
class Config:
    HIDDEN_DIM: int = 64
    RANDOM_SEED: int = 42

@dataclass
class StreamingConfig:
    L_HOPS: int = 2
    MAX_NEIGHBORS_PER_HOP: int = 500
    THRESHOLD_PERCENTILE: float = 95.0
    EMA_ALPHA: float = 0.1
    MAX_TEST_SAMPLES: int = 50000
    CALIBRATION_SAMPLE_SIZE: int = 5000
    BURN_IN_MIN_NEIGHBORS: int = 5
    WELFORD_Z_THRESH: float = 3.0
    COMMUNITY_WELFORD_Z_THRESH: float = 3.0   # Gate 2 — community-pooled Welford tripwire
    COMMUNITY_WELFORD_BURN_IN: int = 5        # min pooled observations before a community baseline is trusted

cfg = Config()
scfg = StreamingConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(cfg.RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.RANDOM_SEED)
np.random.seed(cfg.RANDOM_SEED)
random.seed(cfg.RANDOM_SEED)

INPUT_DIR = Path("/kaggle/input/datasets/kaushik2005/dacisnotebook2")
PROCESSED_DIR = INPUT_DIR / "Processed"
if not PROCESSED_DIR.exists():
    PROCESSED_DIR = Path("./processed")

print(f"Hardware initialization complete. Using device: {device}")
if device.type == "cpu":
    print("WARNING: no GPU attached. MAX_TEST_SAMPLES will truncate the stream to the earliest "
          f"{scfg.MAX_TEST_SAMPLES} test-period transactions chronologically for large graphs — "
          "those numbers are a streaming-subset result, not directly comparable to Notebook 2's "
          "full-test-set metrics. Attach a GPU accelerator if you need the full test set evaluated.")

Hardware initialization complete. Using device: cuda


In [3]:
# Ground-truth copy of the model architecture, augmented with get_embedding()
# for Gate 2's re-embedding step (identical to Notebook 1/2 otherwise).
class DACISGraphSAGE(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int, dropout: float = 0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr='max')
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels, aggr='max')
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        self.linear = nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def get_embedding(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        return x

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        emb = self.get_embedding(x, edge_index)
        emb_drop = F.dropout(emb, p=self.dropout, training=self.training)
        return self.linear(emb_drop)


def best_threshold_for_f1(probs: np.ndarray, labels: np.ndarray) -> float:
    if len(np.unique(labels)) < 2:
        return 0.5
    precision, recall, thresholds = precision_recall_curve(labels, probs)
    f1s = 2 * precision * recall / (precision + recall + 1e-12)
    best_idx = np.nanargmax(f1s[:-1])
    return float(thresholds[best_idx])

## Phase 1 — State Restoration
Fails loudly (rather than silently falling back to an arbitrary order) if the processed graph doesn't carry `time_step` — that means Notebook 1 hasn't been re-run with the timestamp-attaching patch yet.

In [4]:
DATASETS = ["elliptic", "amlsim", "paysim"]

def load_processed_graph(name: str) -> Data:
    path = PROCESSED_DIR / f"{name}_pyg_data.pt"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found.")
    data_dict = torch.load(path, map_location=device, weights_only=False)
    data = Data(**data_dict)
    if not hasattr(data, "time_step") or data.time_step is None:
        raise ValueError(
            f"[{name}] processed graph has no 'time_step' attribute. This notebook requires the "
            f"patched Notebook 1 (prepare_pyg_data now attaches time_step to the Data object) — "
            f"re-run Notebook 1's export cell and re-upload the processed .pt files first."
        )
    return data

def load_trained_model(name: str, in_channels: int) -> DACISGraphSAGE:
    weights_path = INPUT_DIR / f"{name}_best_weights.pt"
    if not weights_path.exists():
        weights_path = Path(".") / f"{name}_best_weights.pt"
    if not weights_path.exists():
        raise FileNotFoundError(f"{weights_path} not found.")

    model = DACISGraphSAGE(in_channels=in_channels, hidden_channels=cfg.HIDDEN_DIM, out_channels=2)
    model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
    model.to(device)
    model.eval()
    return model

assets = {}
for name in DATASETS:
    try:
        data = load_processed_graph(name).to(device)
        model = load_trained_model(name, in_channels=data.num_node_features)
        assets[name] = {"data": data, "model": model}
        print(f"[{name}] restored — nodes={data.num_nodes}, edges={data.edge_index.shape[1]}")
    except (FileNotFoundError, ValueError) as e:
        print(f"[{name}] SKIPPED — {e}")

assert len(assets) > 0, "No datasets restored — check the paths/SKIPPED messages above before continuing."

[elliptic] restored — nodes=203769, edges=234355
[amlsim] restored — nodes=1323234, edges=2646468
[paysim] restored — nodes=1214803, edges=1000000


## Phase 2 — In-Neighbor Adjacency & L-Hop Structural Utilities

`SAGEConv`'s default message flow aggregates a node's embedding from its **in**-neighbors
(source→target), which is the opposite of what `to_scipy_sparse_matrix(edge_index)` gives you by
default (that puts source as the row). Flipping the edge index before building the sparse matrix
gives the correct in-neighbor adjacency — verified: for edges `0->1` and `2->1`, the in-neighbors
of node 1 come back as `[0, 2]`, not `[]`.

This is also where the Structural Invariance Constraint actually gets enforced: the tracker (Phase
3), Gate 2's re-embedding (Phase 4), and this neighborhood definition all now use the *same*
in-neighbor, L-hop, causally-seen-only node set — not three different, inconsistent definitions.

In [5]:
def build_in_neighbor_csr(data: Data):
    """Row i -> in-neighbors of node i (sources of edges pointing into i)."""
    return torch_geometric.utils.to_scipy_sparse_matrix(
        data.edge_index.flip(0), num_nodes=data.num_nodes
    ).tocsr()


def l_hop_in_neighbors(idx: int, in_adj_csr, L: int = 2, max_per_hop: int = 500) -> np.ndarray:
    """BFS over in-neighbors up to L hops — matches an L-layer GraphSAGE's true receptive field."""
    visited = {idx}
    frontier = set(in_adj_csr[idx].indices.tolist())
    frontier -= visited
    all_neighbors = set(frontier)
    visited |= frontier

    for _ in range(L - 1):
        if not frontier:
            break
        next_frontier = set()
        for n in frontier:
            nbrs = in_adj_csr[n].indices
            if len(nbrs) > max_per_hop:
                nbrs = np.random.choice(nbrs, max_per_hop, replace=False)
            next_frontier.update(nbrs.tolist())
        next_frontier -= visited
        all_neighbors |= next_frontier
        visited |= next_frontier
        frontier = next_frontier

    return np.array(sorted(all_neighbors), dtype=np.int64)


def report_isolation_diagnostic(name: str, data: Data, in_adj_csr) -> float:
    """Canary check: what fraction of test nodes have zero real in-neighbors?
    A high fraction here means Gate 2 / the neighbor-based half of Gate 1 will be
    structurally meaningless for this dataset — check upstream graph construction
    before trusting anything downstream."""
    in_degrees = np.diff(in_adj_csr.indptr)
    test_idx_np = data.test_mask.nonzero(as_tuple=False).view(-1).cpu().numpy()
    frac_isolated = (in_degrees[test_idx_np] == 0).mean() if len(test_idx_np) > 0 else 1.0
    print(f"[{name}] fraction of test nodes with zero in-neighbors: {frac_isolated:.2%}")
    if frac_isolated > 0.5:
        print(f"  WARNING: majority of test nodes are structurally isolated for {name} — "
              f"Gate 2 and the local half of Gate 1 will be near-meaningless here. "
              f"Check the upstream edge construction in Notebook 1 before trusting these numbers.")
    return frac_isolated

## Phase 3 — Gate 1 Trackers (magnitude anomaly, OR-gated) + shared Welford primitive

`WelfordTracker` here is the same class Gate 2 reuses in Phase 4, just keyed by a different entity id (account vs. community).

In [6]:
class GlobalDistanceTracker:
    def __init__(self, licit_embeddings: torch.Tensor):
        self.centroid = licit_embeddings.mean(dim=0)

    def score(self, z: torch.Tensor) -> torch.Tensor:
        if z.dim() == 1:
            z = z.unsqueeze(0)
        sim = F.cosine_similarity(z, self.centroid.unsqueeze(0), dim=1)
        return 1.0 - sim


class LocalEMATracker:
    def __init__(self, num_nodes: int, hidden_dim: int, alpha: float = 0.1, min_seen_neighbors: int = 5):
        self.alpha = alpha
        self.ema = torch.zeros((num_nodes, hidden_dim), device=device)
        self.initialized = torch.zeros(num_nodes, dtype=torch.bool, device=device)
        self.min_seen_neighbors = min_seen_neighbors  # burn-in

    def update(self, nodes: torch.Tensor, z: torch.Tensor) -> None:
        uninit = ~self.initialized[nodes]
        init = self.initialized[nodes]

        if uninit.any():
            self.ema[nodes[uninit]] = z[uninit]
            self.initialized[nodes[uninit]] = True

        if init.any():
            self.ema[nodes[init]] = self.alpha * z[init] + (1 - self.alpha) * self.ema[nodes[init]]

    def score(self, node_idx: int, z: torch.Tensor, seen_neighbors: torch.Tensor) -> torch.Tensor:
        # Burn-in: a centroid built from a handful of neighbors is noisy — don't let it
        # trigger an anomaly signal until there's enough neighborhood history behind it.
        if seen_neighbors.numel() < self.min_seen_neighbors:
            return torch.tensor([0.0], device=z.device)

        local_centroid = self.ema[seen_neighbors].mean(dim=0, keepdim=True)
        if z.dim() == 1:
            z = z.unsqueeze(0)
        sim = F.cosine_similarity(z, local_centroid, dim=1)
        return 1.0 - sim

class WelfordTracker:
    """
    Per-entity online mean/variance (Welford's algorithm). Reused by both gates,
    keyed by two different entity spaces:
      - Gate 1: keyed by ACCOUNT node index, not transaction node index, since tx
        nodes are one-shot in this pipeline and never revisited. Only meaningful
        for datasets with sender->tx->receiver routing (AMLSim/PaySim); doesn't
        apply to Elliptic's direct tx-to-tx graph.
      - Gate 2: keyed by COMMUNITY id (a connected component of the structural
        graph — see Phase 4). Many different nodes, including one-shot
        transaction nodes in Elliptic, share the same community, so this still
        builds a real baseline even where per-node or per-account tracking can't.

    Fix vs. the earlier version: an entity's stats are only updated from
    observations NOT flagged by the gate reading them — a flagged observation is
    scored against the baseline but excluded from updating it, so a drifting
    entity can't normalize its own escalation into its own comparison distribution.
    """
    def __init__(self, num_entities: int, hidden_dim: int, burn_in: int = 5):
        self.count = torch.zeros(num_entities, dtype=torch.long, device=device)
        self.mean = torch.zeros((num_entities, hidden_dim), device=device)
        self.M2 = torch.zeros((num_entities, hidden_dim), device=device)
        self.burn_in = burn_in

    def update(self, entity_idx: int, z: torch.Tensor) -> None:
        self.count[entity_idx] += 1
        n = self.count[entity_idx].float()
        delta = z - self.mean[entity_idx]
        self.mean[entity_idx] += delta / n
        delta2 = z - self.mean[entity_idx]
        self.M2[entity_idx] += delta * delta2

    def z_score(self, entity_idx: int, z: torch.Tensor):
        """Mean absolute z-score across hidden dims, or None before burn-in
        (no trustworthy baseline yet — distinct from a z-score of 0)."""
        n = self.count[entity_idx].item()
        if n < self.burn_in:
            return None
        variance = self.M2[entity_idx] / max(n - 1, 1)
        std = torch.sqrt(variance + 1e-8)
        return ((z - self.mean[entity_idx]).abs() / std).mean().item()

    def is_anomaly(self, entity_idx: int, z: torch.Tensor, z_thresh: float = 3.0) -> bool:
        zs = self.z_score(entity_idx, z)
        return zs is not None and zs > z_thresh


def batch_init_group_stats(tracker: "WelfordTracker", group_ids: torch.Tensor, values: torch.Tensor) -> None:
    """Initialize a WelfordTracker's running stats from a whole batch of
    (group_id, value) rows in one vectorized pass.

    Welford's algorithm is just an online way to compute a batch mean/variance —
    its result doesn't depend on the order observations arrive in. So for a
    PRIMING pass over a static historical batch (order genuinely doesn't matter,
    unlike the causal streaming loop where it does), this is exactly equivalent
    to calling .update() once per row, just without an O(n) Python loop over what
    can be a very large training set (e.g. Elliptic, where every node — not just
    a subset with account structure — participates in Gate 2's community prior).
    """
    groups = group_ids.to(device)
    vals = values.to(device)
    num_groups = tracker.mean.shape[0]

    counts = torch.bincount(groups, minlength=num_groups).to(device)
    sums = torch.zeros_like(tracker.mean)
    sums.index_add_(0, groups, vals)

    nonzero = counts > 0
    means = torch.zeros_like(tracker.mean)
    means[nonzero] = sums[nonzero] / counts[nonzero].unsqueeze(1).float()

    centered = vals - means[groups]
    m2 = torch.zeros_like(tracker.mean)
    m2.index_add_(0, groups, centered * centered)

    tracker.count = counts.long()
    tracker.mean = means
    tracker.M2 = m2

## Phase 4 — Gate 2: Community-Level Welford Co-Activation Check

The earlier Structural Invariance Check (re-embedding a node's neighborhood with and without it) was run through the same fraud-vs-legit distribution check used below and showed no meaningful separation — on Elliptic it actively inverted, rejecting every true positive the distance-anomaly fallback found. It's been removed.

Gate 2 now targets a different threat model than Gate 1: individually small, quiet transactions that stay under any single account's own anomaly threshold, but where the surrounding structural neighborhood is collectively drifting. Nodes are grouped into **communities** (weakly-connected components of the graph's structural topology — connectivity only, no label information, so this can't leak fraud signal), and each community gets its own pooled Welford baseline via the *same* `WelfordTracker` class Gate 1 uses, just keyed by community id instead of account id. This is also what lets Gate 2 build a real baseline on Elliptic, where individual transaction nodes are one-shot and never revisited: many different one-shot nodes still land in the same community and feed the same pooled tracker as they stream in.

Gate 2 fires only as a confirmation tier on top of Gate 1 (see Phase 6) — by construction it can only add a confidence tier and a small extra boost, never suppress a Gate 1 catch.

In [7]:
from scipy.sparse.csgraph import breadth_first_order

def build_communities(data: Data, in_adj_csr, max_component_size: int = 500):
    """
    Gate 2's grouping key: weakly-connected components, capped by chunking
    oversized components. Chunks are computed via BFS so members of the
    same chunk are actually topologically close — this is what makes
    "community" mean something relational rather than an arbitrary bucket.

    SAFETY: BFS visitation is validated against connected_components' own
    membership for that component before being trusted. If they disagree,
    that component falls back to sorted-index chunking INSTEAD of silently
    producing corrupted labels — and it's reported explicitly, so you know
    whether your results reflect real locality or the fallback.
    """
    undirected = in_adj_csr.maximum(in_adj_csr.T)
    num_raw_components, raw_labels = connected_components(undirected, directed=False)
    print(f"[diagnostic] raw connected components before capping: {num_raw_components}")

    csr = undirected.tocsr()
    final_labels = np.full(len(raw_labels), -1, dtype=np.int64)
    next_id = 0
    n_bfs_chunked, n_fallback_chunked = 0, 0

    for comp_id in range(num_raw_components):
        members = np.where(raw_labels == comp_id)[0]
        if len(members) <= max_component_size:
            final_labels[members] = next_id
            next_id += 1
            continue

        members_set = set(members.tolist())
        seed = int(members[0])
        order = breadth_first_order(csr, i_start=seed, directed=False,
                                     return_predecessors=False)

        if set(order.tolist()) == members_set:
            chunk_source = order          # genuine BFS locality
            n_bfs_chunked += 1
        else:
            chunk_source = members        # scoped fallback — this component only
            n_fallback_chunked += 1

        for start in range(0, len(chunk_source), max_component_size):
            chunk = chunk_source[start:start + max_component_size]
            final_labels[chunk] = next_id
            next_id += 1

    assert (final_labels >= 0).all(), "BUG: some nodes never got a community id"

    print(f"[diagnostic] oversized components: {n_bfs_chunked} chunked via BFS (topologically local), "
          f"{n_fallback_chunked} chunked via sorted-index fallback (BFS/connected_components disagreed)")

    community_id = torch.as_tensor(final_labels, dtype=torch.long, device=device)
    num_communities = next_id

    sizes = np.bincount(final_labels)
    largest_frac = sizes.max() / len(final_labels)
    print(f"[community structure] {num_communities} communities (capped at {max_component_size}) | "
          f"largest holds {largest_frac:.2%} of nodes | median size {int(np.median(sizes))}")
    if largest_frac > 0.5:
        print("  WARNING: one community still dominates even after capping — "
              "investigate graph connectivity further before trusting Gate 2 here.")
    return community_id, num_communities

def diagnose_community_welford_by_label(dataset_name, data, community_id, community_tracker,
                                         all_embeddings, sample_size=300):
    """Exploratory only — not used for threshold selection. Mirrors the fraud-vs-legit
    distribution check that correctly killed the earlier SIC-based Gate 2: shows whether
    community-pooled Welford z-scores actually separate fraud from legit BEFORE
    gate2_flag is trusted or wired into dg_score anywhere."""
    test_idx = data.test_mask.nonzero(as_tuple=False).view(-1)
    fraud_idx = test_idx[data.y[test_idx] == 1]
    legit_idx = test_idx[data.y[test_idx] == 0]

    if len(fraud_idx) > sample_size:
        fraud_idx = fraud_idx[torch.randperm(len(fraud_idx))[:sample_size]]
    if len(legit_idx) > sample_size:
        legit_idx = legit_idx[torch.randperm(len(legit_idx))[:sample_size]]

    def collect_zscores(node_idx_tensor):
        scores = []
        for v in node_idx_tensor:
            v_item = v.item()
            cid = community_id[v_item].item()
            zs = community_tracker.z_score(cid, all_embeddings[v_item])
            if zs is not None:
                scores.append(zs)
        return np.array(scores)

    fraud_scores = collect_zscores(fraud_idx)
    legit_scores = collect_zscores(legit_idx)

    print(f"\n--- {dataset_name.upper()} Community-Welford z-score by true label (test sample, diagnostic only) ---")
    if len(fraud_scores) > 0:
        print(f"Fraud (n={len(fraud_scores)}): mean={fraud_scores.mean():.4f} | "
              f"median={np.median(fraud_scores):.4f} | p95={np.percentile(fraud_scores, 95):.4f}")
    else:
        print("Fraud: no test nodes with a burned-in community baseline to evaluate.")
    if len(legit_scores) > 0:
        print(f"Legit (n={len(legit_scores)}): mean={legit_scores.mean():.4f} | "
              f"median={np.median(legit_scores):.4f} | p95={np.percentile(legit_scores, 95):.4f}")
    else:
        print("Legit: no test nodes with a burned-in community baseline to evaluate.")

    verdict = "NO SEPARATION — treat Gate 2 as non-functional"
    if len(fraud_scores) > 0 and len(legit_scores) > 0:
        gap = fraud_scores.mean() - legit_scores.mean()
        pooled_std = np.sqrt((fraud_scores.std() ** 2 + legit_scores.std() ** 2) / 2)
        effect_size = gap / pooled_std if pooled_std > 0 else 0.0
        if effect_size > 0.5:
            verdict = f"POSSIBLE SIGNAL — fraud z-scores run higher (effect size {effect_size:.2f})"
        elif effect_size < -0.5:
            verdict = f"INVERTED — fraud z-scores run LOWER than legit (effect size {effect_size:.2f})"
        else:
            verdict = f"NO MEANINGFUL SEPARATION (effect size {effect_size:.2f})"
    print(f"[{dataset_name}] Community-Welford verdict: {verdict}")
    return verdict, fraud_scores, legit_scores

## Phase 5 — Threshold Calibration

`tau_global` and `tau_local` are calibrated the same way as before — the 95th percentile of the corresponding score distribution over validation licit nodes. There's no `tau_sic` anymore: Gate 2's community-Welford check uses a fixed z-score threshold (`COMMUNITY_WELFORD_Z_THRESH`), the same way Gate 1's account-Welford tripwire is already thresholded — both are z-scores against an online per-entity baseline, not raw distances against a global validation distribution, so a percentile calibration doesn't apply to either.

In [8]:
def calibrate_thresholds(data, model, all_embeddings, in_adj_csr, global_tracker, local_tracker):
    licit_val_idx = ((data.y == 0) & data.val_mask).nonzero(as_tuple=False).view(-1)
    if len(licit_val_idx) > scfg.CALIBRATION_SAMPLE_SIZE:
        licit_val_idx = licit_val_idx[torch.randperm(len(licit_val_idx))[:scfg.CALIBRATION_SAMPLE_SIZE]]

    g_dists = [global_tracker.score(all_embeddings[v]).item() for v in licit_val_idx]
    tau_global = float(np.percentile(g_dists, scfg.THRESHOLD_PERCENTILE)) if g_dists else 0.15

    l_dists = []
    for v in licit_val_idx:
        v_item = v.item()
        nbrs = l_hop_in_neighbors(v_item, in_adj_csr, L=scfg.L_HOPS, max_per_hop=scfg.MAX_NEIGHBORS_PER_HOP)
        nbrs_t = torch.tensor(nbrs, dtype=torch.long, device=device)
        seen = nbrs_t[local_tracker.initialized[nbrs_t]] if len(nbrs) > 0 else nbrs_t
        if seen.numel() > 0:
            l_dists.append(local_tracker.score(v_item, all_embeddings[v_item], seen).item())
    tau_local = float(np.percentile(l_dists, scfg.THRESHOLD_PERCENTILE)) if l_dists else tau_global * 1.25

    return tau_global, tau_local

## Phase 6 — Streaming Pipeline

Test-period nodes are processed in true chronological order (`data.time_step`). Gate 1 (account-Welford, where an account entity exists to build a baseline on) is the primary scoring trigger, unchanged from before. Gate 2 (community-Welford) is no longer AND-gated against Gate 1 — it's **tiered on top**: it only ever gets evaluated, and only ever adds a confidence tier (`MODERATE` → `HIGH`) and a small extra score boost, once Gate 1 has already fired. This guarantees, by construction, that every case Gate 1 alone caught before is still caught — Gate 2 can only add information, never subtract a true positive. Gate-purity (precision of each gate, and of the HIGH vs. MODERATE confirmation split) is tracked directly.

In [9]:
def execute_streaming_pipeline(dataset_name, data, model, has_account_structure=False, max_test_samples="default"):
    model.eval()

    sep = "=" * 60
    print(f"\n{sep}\n[STREAMING {dataset_name.upper()}] Precomputing base embeddings & adjacency...\n{sep}")
    with torch.no_grad():
        all_embeddings = model.get_embedding(data.x, data.edge_index)
        all_logits = model.linear(all_embeddings)
        all_probs = F.softmax(all_logits, dim=1)[:, 1]

    in_adj_csr = build_in_neighbor_csr(data)
    report_isolation_diagnostic(dataset_name, data, in_adj_csr)
    community_id, num_communities = build_communities(data, in_adj_csr)

    train_val_mask = data.train_mask | data.val_mask
    licit_train_nodes = (data.y == 0) & train_val_mask

    global_tracker = GlobalDistanceTracker(all_embeddings[licit_train_nodes])
    local_tracker = LocalEMATracker(data.num_nodes, cfg.HIDDEN_DIM, alpha=scfg.EMA_ALPHA, min_seen_neighbors=scfg.BURN_IN_MIN_NEIGHBORS)
    welford_tracker = WelfordTracker(data.num_nodes, cfg.HIDDEN_DIM, burn_in=scfg.BURN_IN_MIN_NEIGHBORS) if has_account_structure else None
    community_tracker = WelfordTracker(num_communities, cfg.HIDDEN_DIM, burn_in=scfg.COMMUNITY_WELFORD_BURN_IN)

    past_nodes = train_val_mask.nonzero(as_tuple=False).view(-1)
    past_order = torch.argsort(data.time_step[past_nodes])
    past_nodes_sorted = past_nodes[past_order]
    local_tracker.update(past_nodes_sorted, all_embeddings[past_nodes_sorted])

    if has_account_structure:
        for v in past_nodes_sorted.tolist():
            in_nbrs = in_adj_csr[v].indices
            if len(in_nbrs) == 1 and data.y[v].item() != 1:  # don't prime baseline from fraud history
                welford_tracker.update(int(in_nbrs[0]), all_embeddings[v])

    licit_past_nodes = past_nodes_sorted[data.y[past_nodes_sorted] != 1]
    batch_init_group_stats(community_tracker, community_id[licit_past_nodes], all_embeddings[licit_past_nodes])

    community_verdict, community_fraud_scores, community_legit_scores = diagnose_community_welford_by_label(
        dataset_name, data, community_id, community_tracker, all_embeddings, sample_size=300
    )

    print("Calibrating thresholds from validation distribution...")
    tau_global, tau_local = calibrate_thresholds(data, model, all_embeddings, in_adj_csr, global_tracker, local_tracker)
    print(f"Calibrated TAU_GLOBAL: {tau_global:.4f} | TAU_LOCAL: {tau_local:.4f}")

    test_idx = data.test_mask.nonzero(as_tuple=False).view(-1)
    time_order = torch.argsort(data.time_step[test_idx])
    test_nodes = test_idx[time_order].tolist()

    if max_test_samples == "default":
        cap = scfg.MAX_TEST_SAMPLES
    else:
        cap = max_test_samples

    if cap is not None and len(test_nodes) > cap:
        print(f"Limiting stream to the earliest {cap} test-period transactions chronologically (of {len(test_nodes)} total).")
        test_nodes = test_nodes[:cap]
    else:
        print(f"Running the FULL test period: {len(test_nodes)} transactions.")

    test_labels_np = data.y[torch.tensor(test_nodes, device=device)].cpu().numpy()
    print(f"[{dataset_name}] test-window fraud prevalence: {(test_labels_np == 1).mean():.4%} "
          f"({(test_labels_np == 1).sum()} / {len(test_labels_np)})")

    y_true, y_pred_baseline, y_pred_dual_gate = [], [], []
    magnitude_triggers, magnitude_tp, magnitude_fp = 0, 0, 0
    welford_triggers, welford_tp, welford_fp = 0, 0, 0
    gate1_triggers, gate1_tp, gate1_fp = 0, 0, 0
    gate2_triggers, gate2_tp, gate2_fp = 0, 0, 0
    welford_hits = []

    # ── NEW: independent test of community-Welford signal, on nodes Gate 1 did
    # NOT flag. Gate 2's is_anomaly() below is only ever called inside the
    # `if gate1_flag:` block — meaning its "100% HIGH-tier confirmation" results
    # seen so far are conditioned entirely on an already-extreme population
    # (accounts Gate 1 already flagged as deviant). That's a selection effect,
    # not evidence of independent coordinated-group detection. This block tests
    # the actual hypothesis: does community z-score separate fraud from legit
    # on the population Gate 1 never touches?
    NEVER_FLAGGED_SAMPLE_CAP = 300
    never_flagged_fraud_z, never_flagged_legit_z = [], []

    torch.cuda.empty_cache()

    for idx in tqdm(test_nodes, desc=f"Processing {dataset_name} Stream"):
        z_v = all_embeddings[idx]
        prob_base = all_probs[idx].item()
        label = data.y[idx].item()

        y_true.append(label)
        y_pred_baseline.append(prob_base)

        candidate_neighbors = l_hop_in_neighbors(idx, in_adj_csr, L=scfg.L_HOPS, max_per_hop=scfg.MAX_NEIGHBORS_PER_HOP)
        cand_t = torch.tensor(candidate_neighbors, dtype=torch.long, device=device)
        seen_neighbors = cand_t[local_tracker.initialized[cand_t]] if len(candidate_neighbors) > 0 else cand_t

        g_dist = global_tracker.score(z_v).item()
        l_dist = local_tracker.score(idx, z_v, seen_neighbors).item() if seen_neighbors.numel() > 0 else 0.0

        sender_account_idx = None
        welford_flag = False
        account_n_at_trigger = None
        if has_account_structure:
            in_nbrs = in_adj_csr[idx].indices
            if len(in_nbrs) == 1:
                sender_account_idx = int(in_nbrs[0])
                account_n_at_trigger = welford_tracker.count[sender_account_idx].item()
                welford_flag = welford_tracker.is_anomaly(sender_account_idx, z_v, z_thresh=scfg.WELFORD_Z_THRESH)

        magnitude_flag = (g_dist > tau_global) or (l_dist > tau_local)
        if magnitude_flag and seen_neighbors.numel() > 0:
            magnitude_triggers += 1
            magnitude_tp += int(label == 1)
            magnitude_fp += int(label == 0)

        gate1_flag = welford_flag

        # ── NEW: sample community z-score for nodes Gate 1 did NOT flag,
        # split by true label, BEFORE this node's own update() call below —
        # same causal read-then-update ordering used everywhere else here.
        if not gate1_flag:
            cid_check = community_id[idx].item()
            zc = community_tracker.z_score(cid_check, z_v)
            if zc is not None:
                if label == 1 and len(never_flagged_fraud_z) < NEVER_FLAGGED_SAMPLE_CAP:
                    never_flagged_fraud_z.append(zc)
                elif label == 0 and len(never_flagged_legit_z) < NEVER_FLAGGED_SAMPLE_CAP:
                    never_flagged_legit_z.append(zc)

        gate2_flag = False
        confidence_tier = None
        dg_score = prob_base

        if gate1_flag:
            welford_triggers += 1
            welford_tp += int(label == 1)
            welford_fp += int(label == 0)
            welford_hits.append({
                "node": idx, "account": sender_account_idx, "label": label,
                "prob_base": prob_base, "account_n_at_trigger": account_n_at_trigger,
            })

            gate1_triggers += 1
            gate1_tp += int(label == 1)
            gate1_fp += int(label == 0)

            dg_score = min(1.0, dg_score * 1.5)

            cid = community_id[idx].item()
            gate2_flag = community_tracker.is_anomaly(cid, z_v, z_thresh=scfg.COMMUNITY_WELFORD_Z_THRESH)
            if gate2_flag:
                dg_score = min(1.0, dg_score * 1.2)
                confidence_tier = "HIGH"
                gate2_triggers += 1
                gate2_tp += int(label == 1)
                gate2_fp += int(label == 0)
            else:
                confidence_tier = "MODERATE"

        y_pred_dual_gate.append(dg_score)
        local_tracker.update(torch.tensor([idx], device=device), z_v.unsqueeze(0))

        if sender_account_idx is not None and not gate1_flag:
            welford_tracker.update(sender_account_idx, z_v)
        if not gate2_flag:
            community_tracker.update(community_id[idx].item(), z_v)

    # ── NEW: compute the independent (unconditioned) verdict ────────────────
    never_flagged_verdict = "INSUFFICIENT SAMPLE — too few burned-in nodes outside Gate 1's catches"
    ind_effect_size = None
    if len(never_flagged_fraud_z) >= 10 and len(never_flagged_legit_z) >= 10:
        f_arr = np.array(never_flagged_fraud_z)
        l_arr = np.array(never_flagged_legit_z)
        gap = f_arr.mean() - l_arr.mean()
        pooled_std = np.sqrt((f_arr.std()**2 + l_arr.std()**2) / 2)
        ind_effect_size = gap / pooled_std if pooled_std > 0 else 0.0
        if ind_effect_size > 0.5:
            never_flagged_verdict = f"POSSIBLE INDEPENDENT SIGNAL (effect size {ind_effect_size:.2f})"
        elif ind_effect_size < -0.5:
            never_flagged_verdict = f"INVERTED (effect size {ind_effect_size:.2f})"
        else:
            never_flagged_verdict = f"NO INDEPENDENT SEPARATION (effect size {ind_effect_size:.2f})"

    print(f"\n--- {dataset_name.upper()} Gate Trigger & Purity Diagnostics ---")

    if has_account_structure:
        print(f"GATE 1 — Account Welford Tripwire  [SCORES dg_score]")
        print(f"  Fired: {welford_triggers} / {len(test_nodes)}")
        if welford_triggers > 0:
            print(f"  Precision: {welford_tp / welford_triggers:.4f}  "
                  f"({welford_tp} fraud, {welford_fp} legit)")
    else:
        print(f"GATE 1 — Account Welford Tripwire  [NOT APPLICABLE]")
        print(f"  No recurring account structure in this dataset — Gate 1 cannot")
        print(f"  build a per-entity baseline. dg_score == prob_base for all nodes.")

    print(f"\nDistance-Anomaly Check (global/local)  [non-scoring — candidate fallback signal]")
    print(f"  Fired: {magnitude_triggers} / {len(test_nodes)}")
    if magnitude_triggers > 0:
        print(f"  Precision: {magnitude_tp / magnitude_triggers:.4f}  ({magnitude_tp} fraud, {magnitude_fp} legit)")

    print(f"\nGATE 2 — Community-Welford Co-Activation Check  [tiered confirmation — SCORES dg_score via confidence tier]")
    print(f"  Community-Welford verdict (unconditioned diagnostic sample, see above): {community_verdict}")
    print(f"  Community-Welford verdict (population Gate 1 NEVER flagged, n_fraud={len(never_flagged_fraud_z)}, "
          f"n_legit={len(never_flagged_legit_z)}): {never_flagged_verdict}")
    if gate1_triggers > 0:
        moderate_triggers = gate1_triggers - gate2_triggers
        moderate_tp = gate1_tp - gate2_tp
        moderate_fp = gate1_fp - gate2_fp
        print(f"  Gate 1 alone (unchanged): {gate1_triggers} fired, "
              f"precision {gate1_tp / gate1_triggers:.4f}  ({gate1_tp} fraud, {gate1_fp} legit)")
        print(f"  ── split by confidence tier (CAUTION: computed only on the Gate-1-flagged")
        print(f"     population — see 'never flagged' verdict above for the unconditioned test) ──")
        print(f"  HIGH   (Gate 1 + Gate 2 confirmed): {gate2_triggers} / {gate1_triggers}")
        if gate2_triggers > 0:
            print(f"    Precision: {gate2_tp / gate2_triggers:.4f}  ({gate2_tp} fraud, {gate2_fp} legit)")
        print(f"  MODERATE (Gate 1 only, not community-confirmed): {moderate_triggers} / {gate1_triggers}")
        if moderate_triggers > 0:
            print(f"    Precision: {moderate_tp / moderate_triggers:.4f}  ({moderate_tp} fraud, {moderate_fp} legit)")
    else:
        print(f"  Gate 1 never fired in this stream — no Gate-1-triggered subset for Gate 2 to confirm against.")

    val_idx = data.val_mask.nonzero(as_tuple=False).view(-1)
    y_true_val = data.y[val_idx].cpu().numpy()
    y_pred_val = all_probs[val_idx].detach().cpu().numpy()

    return {
        "y_true_val": y_true_val, "y_pred_val": y_pred_val,
        "y_true_test": np.array(y_true), "y_pred_baseline": np.array(y_pred_baseline),
        "y_pred_dual_gate": np.array(y_pred_dual_gate), "welford_hits": welford_hits,
        "community_verdict": community_verdict,
        "community_fraud_scores": community_fraud_scores, "community_legit_scores": community_legit_scores,
        "never_flagged_verdict": never_flagged_verdict, "never_flagged_effect_size": ind_effect_size,
        "never_flagged_fraud_z": never_flagged_fraud_z, "never_flagged_legit_z": never_flagged_legit_z,
        "gate1_triggers": gate1_triggers, "gate1_tp": gate1_tp, "gate1_fp": gate1_fp,
        "gate2_triggers": gate2_triggers, "gate2_tp": gate2_tp, "gate2_fp": gate2_fp,
    }

In [10]:
def evaluate_predictions(y_true_val, y_pred_val, y_true_test, y_pred_test, name=""):
    if len(np.unique(y_true_test)) < 2:
        print(f"[{name}] test set has only one class present in this stream — skipping.")
        return {"auroc": 0.5, "auprc": 0.0, "f1": 0.0, "threshold": 0.5}

    auroc = roc_auc_score(y_true_test, y_pred_test)
    precision, recall, _ = precision_recall_curve(y_true_test, y_pred_test)
    auprc = auc(recall, precision)

    # FIX: threshold tuned on VAL, applied to TEST — no more optimizing directly on test labels.
    best_thresh = best_threshold_for_f1(y_pred_val, y_true_val) if len(np.unique(y_true_val)) >= 2 else 0.5
    preds = (y_pred_test >= best_thresh).astype(int)
    f1 = f1_score(y_true_test, preds, zero_division=0)

    print(f"[{name}] AUROC: {auroc:.4f} | AUPRC: {auprc:.4f} | F1@val-tuned-threshold({best_thresh:.3f}): {f1:.4f}")
    return {"auroc": auroc, "auprc": auprc, "f1": f1, "threshold": best_thresh}

## Execute — Dual-Gate vs Baseline, per dataset

In [11]:
results = {}
for name in assets:
    data = assets[name]["data"]
    model = assets[name]["model"]

    max_samples = None if name == "amlsim" else scfg.MAX_TEST_SAMPLES
    out = execute_streaming_pipeline(name, data, model, has_account_structure=(name in ("amlsim", "paysim")), max_test_samples=max_samples)
    results[name] = out

    print(f"\n--- {name.upper()} Dual-Gate System Performance vs Baseline (same val-tuned threshold for both) ---")
    baseline_metrics = evaluate_predictions(out["y_true_val"], out["y_pred_val"], out["y_true_test"], out["y_pred_baseline"], name=f"{name}-Baseline")
    dual_gate_metrics = evaluate_predictions(out["y_true_val"], out["y_pred_val"], out["y_true_test"], out["y_pred_dual_gate"], name=f"{name}-DualGate")
    print()
    if name == "amlsim":
        results[name]["threshold"] = dual_gate_metrics["threshold"]


[STREAMING ELLIPTIC] Precomputing base embeddings & adjacency...
[elliptic] fraction of test nodes with zero in-neighbors: 36.23%
[diagnostic] raw connected components before capping: 49
[diagnostic] oversized components: 49 chunked via BFS (topologically local), 0 chunked via sorted-index fallback (BFS/connected_components disagreed)
[community structure] 433 communities (capped at 500) | largest holds 0.25% of nodes | median size 500

--- ELLIPTIC Community-Welford z-score by true label (test sample, diagnostic only) ---
Fraud: no test nodes with a burned-in community baseline to evaluate.
Legit: no test nodes with a burned-in community baseline to evaluate.
[elliptic] Community-Welford verdict: NO SEPARATION — treat Gate 2 as non-functional
Calibrating thresholds from validation distribution...
Calibrated TAU_GLOBAL: 0.8857 | TAU_LOCAL: 0.4360
Running the FULL test period: 11184 transactions.
[elliptic] test-window fraud prevalence: 5.6867% (636 / 11184)


Processing elliptic Stream:   0%|          | 0/11184 [00:00<?, ?it/s]


--- ELLIPTIC Gate Trigger & Purity Diagnostics ---
GATE 1 — Account Welford Tripwire  [NOT APPLICABLE]
  No recurring account structure in this dataset — Gate 1 cannot
  build a per-entity baseline. dg_score == prob_base for all nodes.

Distance-Anomaly Check (global/local)  [non-scoring — candidate fallback signal]
  Fired: 775 / 11184
  Precision: 0.0761  (59 fraud, 716 legit)

GATE 2 — Community-Welford Co-Activation Check  [tiered confirmation — SCORES dg_score via confidence tier]
  Community-Welford verdict (unconditioned diagnostic sample, see above): NO SEPARATION — treat Gate 2 as non-functional
  Community-Welford verdict (population Gate 1 NEVER flagged, n_fraud=300, n_legit=300): NO INDEPENDENT SEPARATION (effect size -0.22)
  Gate 1 never fired in this stream — no Gate-1-triggered subset for Gate 2 to confirm against.

--- ELLIPTIC Dual-Gate System Performance vs Baseline (same val-tuned threshold for both) ---
[elliptic-Baseline] AUROC: 0.8112 | AUPRC: 0.4388 | F1@val-tu

Processing amlsim Stream:   0%|          | 0/266199 [00:00<?, ?it/s]


--- AMLSIM Gate Trigger & Purity Diagnostics ---
GATE 1 — Account Welford Tripwire  [SCORES dg_score]
  Fired: 380 / 266199
  Precision: 0.9921  (377 fraud, 3 legit)

Distance-Anomaly Check (global/local)  [non-scoring — candidate fallback signal]
  Fired: 25761 / 266199
  Precision: 0.0024  (61 fraud, 25700 legit)

GATE 2 — Community-Welford Co-Activation Check  [tiered confirmation — SCORES dg_score via confidence tier]
  Community-Welford verdict (unconditioned diagnostic sample, see above): NO MEANINGFUL SEPARATION (effect size 0.28)
  Community-Welford verdict (population Gate 1 NEVER flagged, n_fraud=0, n_legit=300): INSUFFICIENT SAMPLE — too few burned-in nodes outside Gate 1's catches
  Gate 1 alone (unchanged): 380 fired, precision 0.9921  (377 fraud, 3 legit)
  ── split by confidence tier (CAUTION: computed only on the Gate-1-flagged
     population — see 'never flagged' verdict above for the unconditioned test) ──
  HIGH   (Gate 1 + Gate 2 confirmed): 33 / 380
    Precision

Processing paysim Stream:   0%|          | 0/50000 [00:00<?, ?it/s]


--- PAYSIM Gate Trigger & Purity Diagnostics ---
GATE 1 — Account Welford Tripwire  [SCORES dg_score]
  Fired: 0 / 50000

Distance-Anomaly Check (global/local)  [non-scoring — candidate fallback signal]
  Fired: 0 / 50000

GATE 2 — Community-Welford Co-Activation Check  [tiered confirmation — SCORES dg_score via confidence tier]
  Community-Welford verdict (unconditioned diagnostic sample, see above): NO MEANINGFUL SEPARATION (effect size -0.24)
  Community-Welford verdict (population Gate 1 NEVER flagged, n_fraud=3, n_legit=300): INSUFFICIENT SAMPLE — too few burned-in nodes outside Gate 1's catches
  Gate 1 never fired in this stream — no Gate-1-triggered subset for Gate 2 to confirm against.

--- PAYSIM Dual-Gate System Performance vs Baseline (same val-tuned threshold for both) ---
[paysim-Baseline] AUROC: 0.8007 | AUPRC: 0.0009 | F1@val-tuned-threshold(0.889): 0.0000
[paysim-DualGate] AUROC: 0.8007 | AUPRC: 0.0009 | F1@val-tuned-threshold(0.889): 0.0000



In [12]:
# ── Diagnostic: is len(in_nbrs)==1 rare, and is it fraud-specific? ──────────────
# Checks whether Gate 1's account-identification step is picking up a genuine pattern
# or an incidental artifact, before AMLSim's 377/377-recall result goes in the paper.

for ds_name in ("amlsim", "paysim"):
    if ds_name not in assets:
        continue
    data = assets[ds_name]["data"]
    in_adj_csr = build_in_neighbor_csr(data)
    test_idx = data.test_mask.nonzero(as_tuple=False).view(-1).cpu().numpy()
    in_degrees = np.diff(in_adj_csr.indptr)[test_idx]
    labels = data.y[torch.tensor(test_idx, device=device)].cpu().numpy()
    overall_fraud_rate = (labels == 1).mean()

    print(f"\n--- {ds_name.upper()} in-neighbor degree distribution (test period, overall fraud rate {overall_fraud_rate*100:.3f}%) ---")
    for deg in sorted(np.unique(in_degrees))[:6]:
        mask = in_degrees == deg
        n = int(mask.sum())
        n_fraud = int((labels[mask] == 1).sum())
        print(f"  in-degree {deg}: {n:7,d} nodes  ({n_fraud} fraud, {n_fraud/n*100:.3f}% fraud rate)")

    n_single = int((in_degrees == 1).sum())
    print(f"  -> {n_single:,} / {len(test_idx):,} test nodes ({n_single/len(test_idx)*100:.3f}%) have exactly 1 in-neighbor")


--- AMLSIM in-neighbor degree distribution (test period, overall fraud rate 0.142%) ---
  in-degree 1: 266,199 nodes  (377 fraud, 0.142% fraud rate)
  -> 266,199 / 266,199 test nodes (100.000%) have exactly 1 in-neighbor

--- PAYSIM in-neighbor degree distribution (test period, overall fraud rate 0.024%) ---
  in-degree 1: 151,809 nodes  (36 fraud, 0.024% fraud rate)
  -> 151,809 / 151,809 test nodes (100.000%) have exactly 1 in-neighbor


In [13]:
hits = results["amlsim"]["welford_hits"]
threshold = results["amlsim"]["threshold"]

fraud_hits = [h for h in hits if h["label"] == 1]
distinct_accounts = {h["account"] for h in fraud_hits}

rescued = [h for h in fraud_hits if h["prob_base"] < threshold]         # was a baseline FN, boost fixed it
already_correct = [h for h in fraud_hits if h["prob_base"] >= threshold] # baseline already had it right

near_burn_in = [h for h in fraud_hits if h["account_n_at_trigger"] is not None and h["account_n_at_trigger"] < 15]

print(f"AMLSim — Welford fraud catches: {len(fraud_hits)}")
print(f"  Distinct sender accounts responsible: {len(distinct_accounts)}  "
      f"({len(fraud_hits)/max(len(distinct_accounts),1):.1f} caught transactions per account on average)")
print(f"  Genuinely rescued (baseline had it below threshold): {len(rescued)} / {len(fraud_hits)}")
print(f"  Already correct at baseline (boost was redundant here): {len(already_correct)} / {len(fraud_hits)}")
print(f"  Triggered on an account with <15 prior transactions (fragile variance estimate): {len(near_burn_in)} / {len(fraud_hits)}")

AMLSim — Welford fraud catches: 377
  Distinct sender accounts responsible: 359  (1.1 caught transactions per account on average)
  Genuinely rescued (baseline had it below threshold): 205 / 377
  Already correct at baseline (boost was redundant here): 172 / 377
  Triggered on an account with <15 prior transactions (fragile variance estimate): 20 / 377


In [14]:
# Calculate total fraud across AMLSim's entire dataset (not the stale loop variable)
amlsim_data = assets["amlsim"]["data"]
total_fraud_in_dataset = (amlsim_data.y == 1).sum().item()
print(f"Total actual fraud in the entire AMLSim dataset: {total_fraud_in_dataset}")

Total actual fraud in the entire AMLSim dataset: 1719


In [15]:
# ── IEEE-CIS: load and temporally split ─────────────────────────────────────
#
# Only train_transaction.csv has real isFraud labels (test_transaction.csv is
# the unlabeled Kaggle submission target — not usable for evaluation here).
# We build our own chronological train/val/test split from TransactionDT,
# same pattern as AMLSim/PaySim.

IEEECIS_DIR = Path("/kaggle/input/competitions/ieee-fraud-detection")

ieee_raw = pd.read_csv(IEEECIS_DIR / "train_transaction.csv")
print(f"IEEE-CIS raw transactions: {len(ieee_raw):,}")
print(f"Fraud rate: {ieee_raw['isFraud'].mean()*100:.3f}%")
print(f"Unique card1 accounts: {ieee_raw['card1'].nunique():,}")

ieee_raw = ieee_raw.sort_values("TransactionDT").reset_index(drop=True)

n = len(ieee_raw)
train_cut = int(n * 0.70)
val_cut   = int(n * 0.85)

ieee_train = ieee_raw.iloc[:train_cut].copy()
ieee_val   = ieee_raw.iloc[train_cut:val_cut].copy()
ieee_test  = ieee_raw.iloc[val_cut:].copy()

print(f"\nTemporal split: train={len(ieee_train):,}  val={len(ieee_val):,}  test={len(ieee_test):,}")
print(f"Test fraud rate: {ieee_test['isFraud'].mean()*100:.3f}%  "
      f"({ieee_test['isFraud'].sum():,} fraud)")

IEEE-CIS raw transactions: 590,540
Fraud rate: 3.499%
Unique card1 accounts: 13,553

Temporal split: train=413,378  val=88,581  test=88,581
Test fraud rate: 3.480%  (3,083 fraud)


In [16]:
# ── Composite entity key (replaces card1 alone) ──────────────────────────────
# card1+card2+addr1 landed at median entity size 2, mean 14.2 — a believable
# real-world usage pattern, avoiding both card1-alone's collision problem
# (one 14,932-txn outlier bucket) and card1+card2+addr1+D1's over-splitting
# (median size 1, no recurrence at all).

for df in (ieee_train, ieee_val, ieee_test, ieee_raw):
    df["entity_key"] = (
        df["card1"].astype(str) + "_" +
        df["card2"].fillna(-1).astype(str) + "_" +
        df["addr1"].fillna(-1).astype(str)
    )

print(f"Entities: {ieee_raw['entity_key'].nunique():,}")
print(f"Median entity size: {ieee_raw.groupby('entity_key').size().median():.0f}")

Entities: 41,672
Median entity size: 2


In [17]:
# ── IEEE-CIS: bipartite account→transaction graph ───────────────────────────
#
# Node layout: [account nodes] followed by [transaction nodes].
# Each transaction gets edges in BOTH directions to its entity_key account —
# account->txn (so a transaction's embedding aggregates from its account, as
# before) AND txn->account (so the account itself has in-neighbors). Without
# the second direction, accounts have zero in-neighbors, which makes the
# 2-hop BFS Gate 1/2 and the local tracker rely on structurally unable to
# reach past the account — every transaction's "L-hop neighborhood" collapses
# to just {its account}, and no sibling transaction is ever reachable. This
# matches the bidirectional route-through convention AMLSim/PaySim already use.

all_accounts = pd.concat([ieee_train[["entity_key"]], ieee_val[["entity_key"]], ieee_test[["entity_key"]]]).drop_duplicates()
IEEE_ACCT_TO_IDX = {acc: i for i, acc in enumerate(all_accounts["entity_key"].values)}
N_ACCOUNTS = len(IEEE_ACCT_TO_IDX)

def build_ieee_split(df: pd.DataFrame, node_offset: int):
    """Assigns transaction node indices starting at node_offset (after all
    account nodes), builds edges in both directions between each transaction
    and its account."""
    n_txn = len(df)
    txn_idx = np.arange(node_offset, node_offset + n_txn)
    acct_idx = df["entity_key"].map(IEEE_ACCT_TO_IDX).values
    edge_src = np.concatenate([acct_idx, txn_idx])   # account->txn, then txn->account
    edge_dst = np.concatenate([txn_idx, acct_idx])
    return txn_idx, np.vstack([edge_src, edge_dst])

TXN_OFFSET = N_ACCOUNTS
train_txn_idx, train_edges = build_ieee_split(ieee_train, TXN_OFFSET)
val_txn_idx,   val_edges   = build_ieee_split(ieee_val,   TXN_OFFSET + len(ieee_train))
test_txn_idx,  test_edges  = build_ieee_split(ieee_test,  TXN_OFFSET + len(ieee_train) + len(ieee_val))

N_TXN = len(ieee_train) + len(ieee_val) + len(ieee_test)
N_TOTAL = N_ACCOUNTS + N_TXN
edge_index = np.hstack([train_edges, val_edges, test_edges])

print(f"Accounts: {N_ACCOUNTS:,}  Transactions: {N_TXN:,}  Total nodes: {N_TOTAL:,}")
print(f"Edges: {edge_index.shape[1]:,} (bidirectional — was {edge_index.shape[1]//2:,} one-directional before this fix)")

Accounts: 41,672  Transactions: 590,540  Total nodes: 632,212
Edges: 1,181,080 (bidirectional — was 590,540 one-directional before this fix)


In [18]:
# ── IEEE-CIS: expanded feature set — V-block (PCA), categoricals, match flags,
# and time-of-day. Previous feature set (~30 dims, amount + C/D only) excluded
# the columns IEEE-CIS's predictive signal actually concentrates in.

from sklearn.decomposition import PCA

# 1. V-block: 339 highly correlated, sparse columns -- PCA-reduce to 25 dims,
#    fit on TRAIN only to avoid leakage into val/test.
V_COLS = [c for c in ieee_raw.columns if c.startswith("V")]
print(f"V-columns found: {len(V_COLS)}")

v_train_raw = ieee_train[V_COLS].fillna(ieee_train[V_COLS].median(numeric_only=True)).fillna(0.0)
v_val_raw   = ieee_val[V_COLS].fillna(ieee_train[V_COLS].median(numeric_only=True)).fillna(0.0)
v_test_raw  = ieee_test[V_COLS].fillna(ieee_train[V_COLS].median(numeric_only=True)).fillna(0.0)

v_pca = PCA(n_components=25, random_state=42)
v_train_pca = v_pca.fit_transform(v_train_raw)
v_val_pca   = v_pca.transform(v_val_raw)
v_test_pca  = v_pca.transform(v_test_raw)
print(f"V-block PCA: {len(V_COLS)} -> 25 dims, explained variance: {v_pca.explained_variance_ratio_.sum():.3f}")

# 2. Categorical identity fields -- frequency-encode, computed on TRAIN only.
CAT_COLS = ["ProductCD", "card4", "card6", "P_emaildomain", "DeviceType"]
CAT_COLS = [c for c in CAT_COLS if c in ieee_raw.columns]

freq_maps = {c: ieee_train[c].value_counts(normalize=True).to_dict() for c in CAT_COLS}

def encode_categoricals(df):
    return np.column_stack([
        df[c].map(freq_maps[c]).fillna(0.0).values for c in CAT_COLS
    ]).astype(np.float32)

cat_train = encode_categoricals(ieee_train)
cat_val   = encode_categoricals(ieee_val)
cat_test  = encode_categoricals(ieee_test)

# 3. M1-M9 match flags -- T/F/NaN -> 1/0/-1.
M_COLS = [f"M{i}" for i in range(1, 10) if f"M{i}" in ieee_raw.columns]

def encode_match_flags(df):
    out = np.zeros((len(df), len(M_COLS)), dtype=np.float32)
    for j, c in enumerate(M_COLS):
        out[:, j] = df[c].map({"T": 1.0, "F": 0.0}).fillna(-1.0).values
    return out

m_train = encode_match_flags(ieee_train)
m_val   = encode_match_flags(ieee_val)
m_test  = encode_match_flags(ieee_test)

# 4. Hour-of-day, cyclically encoded (fraud shows real diurnal patterns).
def encode_hour(df):
    hour = (df["TransactionDT"] // 3600) % 24
    return np.column_stack([np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24)]).astype(np.float32)

hour_train = encode_hour(ieee_train)
hour_val   = encode_hour(ieee_val)
hour_test  = encode_hour(ieee_test)

# ── Original amount/C/D features ─────────────────────────────────────────────
BASE_FEAT_COLS = ["TransactionAmt"] + [f"C{i}" for i in range(1, 15)] + [f"D{i}" for i in range(1, 16)]
BASE_FEAT_COLS = [c for c in BASE_FEAT_COLS if c in ieee_raw.columns]

def make_base_features(df):
    x = df[BASE_FEAT_COLS].copy()
    return x.fillna(x.median(numeric_only=True)).fillna(0.0).values.astype(np.float32)

base_train = make_base_features(ieee_train)
base_val   = make_base_features(ieee_val)
base_test  = make_base_features(ieee_test)

# ── Concatenate everything into the final transaction feature matrix ────────
txn_x_train = np.hstack([base_train, v_train_pca, cat_train, m_train, hour_train]).astype(np.float32)
txn_x_val   = np.hstack([base_val,   v_val_pca,   cat_val,   m_val,   hour_val]).astype(np.float32)
txn_x_test  = np.hstack([base_test,  v_test_pca,  cat_test,  m_test,  hour_test]).astype(np.float32)
N_TXN_FEAT = txn_x_train.shape[1]

print(f"Final feature dim: {N_TXN_FEAT}  "
      f"(base {base_train.shape[1]} + V-PCA 25 + categorical {cat_train.shape[1]} + "
      f"match-flags {m_train.shape[1]} + hour 2)")

# ── Account full-history aggregates (unchanged, still entity_key-keyed) ─────
acct_agg = ieee_raw.groupby("entity_key").agg(
    tx_count     = ("TransactionAmt", "count"),
    total_amount = ("TransactionAmt", "sum"),
    avg_amount   = ("TransactionAmt", "mean"),
    max_amount   = ("TransactionAmt", "max"),
    std_amount   = ("TransactionAmt", "std"),
).reset_index()
acct_agg["std_amount"] = acct_agg["std_amount"].fillna(0.0)
acct_agg["acct_idx"] = acct_agg["entity_key"].map(IEEE_ACCT_TO_IDX)
acct_agg = acct_agg.sort_values("acct_idx")

ACCT_FEAT_COLS = ["tx_count", "total_amount", "avg_amount", "max_amount", "std_amount"]
N_ACCT_FEAT = len(ACCT_FEAT_COLS)

acct_x = np.zeros((N_ACCOUNTS, N_TXN_FEAT), dtype=np.float32)
acct_x[:, :N_ACCT_FEAT] = acct_agg[ACCT_FEAT_COLS].values.astype(np.float32)

print(f"Account features padded into {N_TXN_FEAT}-dim width")

V-columns found: 339
V-block PCA: 339 -> 25 dims, explained variance: 1.000
Final feature dim: 70  (base 30 + V-PCA 25 + categorical 4 + match-flags 9 + hour 2)
Account features padded into 70-dim width


In [19]:
from sklearn.preprocessing import StandardScaler

# ── IEEE-CIS: assemble PyG Data, scaled and masked ──────────────────────────

full_x = np.zeros((N_TOTAL, N_TXN_FEAT), dtype=np.float32)
full_x[:N_ACCOUNTS] = acct_x
full_x[train_txn_idx] = txn_x_train
full_x[val_txn_idx]   = txn_x_val
full_x[test_txn_idx]  = txn_x_test

# Scale using TRAIN-split transaction stats only (avoid leaking val/test scale)
ieee_scaler = StandardScaler()
ieee_scaler.fit(txn_x_train)
full_x[:N_ACCOUNTS]    = ieee_scaler.transform(full_x[:N_ACCOUNTS])
full_x[train_txn_idx]  = ieee_scaler.transform(full_x[train_txn_idx])
full_x[val_txn_idx]    = ieee_scaler.transform(full_x[val_txn_idx])
full_x[test_txn_idx]   = ieee_scaler.transform(full_x[test_txn_idx])

full_y = np.full(N_TOTAL, -1, dtype=np.int64)   # -1 for account nodes (unlabeled)
full_y[train_txn_idx] = ieee_train["isFraud"].values
full_y[val_txn_idx]   = ieee_val["isFraud"].values
full_y[test_txn_idx]  = ieee_test["isFraud"].values

full_time = np.zeros(N_TOTAL, dtype=np.int64)
full_time[train_txn_idx] = ieee_train["TransactionDT"].values
full_time[val_txn_idx]   = ieee_val["TransactionDT"].values
full_time[test_txn_idx]  = ieee_test["TransactionDT"].values

train_mask = np.zeros(N_TOTAL, dtype=bool); train_mask[train_txn_idx] = True
val_mask   = np.zeros(N_TOTAL, dtype=bool); val_mask[val_txn_idx]     = True
test_mask  = np.zeros(N_TOTAL, dtype=bool); test_mask[test_txn_idx]   = True

ieeecis_data = Data(
    x=torch.tensor(full_x, dtype=torch.float),
    edge_index=torch.tensor(edge_index, dtype=torch.long),
    y=torch.tensor(full_y, dtype=torch.long),
    time_step=torch.tensor(full_time, dtype=torch.long),
    train_mask=torch.tensor(train_mask),
    val_mask=torch.tensor(val_mask),
    test_mask=torch.tensor(test_mask),
).to(device)

print(f"IEEE-CIS Data object: {ieeecis_data.num_nodes:,} nodes, {ieeecis_data.num_edges:,} edges")
print(f"Feature dim: {ieeecis_data.x.shape[1]}")

IEEE-CIS Data object: 632,212 nodes, 1,181,080 edges
Feature dim: 70


In [20]:
# ── IEEE-CIS: train baseline DACISGraphSAGE ──────────────────────────────────
# Reuses the exact model class already defined in Cell 3. ~600K total nodes
# — comparable to Elliptic's scale, should fit full-batch; drop to mini-batch
# NeighborLoader (same pattern used for PaySim) if you hit an OOM here.

ieee_labeled_train = train_mask & (full_y != -1)
class_counts = np.bincount(full_y[ieee_labeled_train])
class_weights = torch.tensor(class_counts.max() / class_counts, dtype=torch.float, device=device)

ieee_model = DACISGraphSAGE(
    in_channels=N_TXN_FEAT,
    hidden_channels=cfg.HIDDEN_DIM,
    out_channels=2,
).to(device)
optimizer = torch.optim.AdamW(ieee_model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(weight=class_weights)

train_idx_t = torch.tensor(train_txn_idx, device=device)
val_idx_t   = torch.tensor(val_txn_idx, device=device)

best_val_auc, best_state, patience_ctr = 0.0, None, 0

for epoch in range(1, 101):
    ieee_model.train()
    optimizer.zero_grad()
    logits = ieee_model(ieeecis_data.x, ieeecis_data.edge_index)
    loss = criterion(logits[train_idx_t], ieeecis_data.y[train_idx_t])
    loss.backward()
    nn.utils.clip_grad_norm_(ieee_model.parameters(), max_norm=1.0)
    optimizer.step()

    ieee_model.eval()
    with torch.no_grad():
        val_logits = ieee_model(ieeecis_data.x, ieeecis_data.edge_index)
        val_probs = F.softmax(val_logits[val_idx_t], dim=1)[:, 1].cpu().numpy()
        val_auc = roc_auc_score(ieeecis_data.y[val_idx_t].cpu().numpy(), val_probs)

    if val_auc > best_val_auc:
        best_val_auc, best_state, patience_ctr = val_auc, {k: v.clone() for k, v in ieee_model.state_dict().items()}, 0
    else:
        patience_ctr += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | loss {loss.item():.4f} | val AUROC {val_auc:.4f}")
    if patience_ctr >= 15:
        print(f"Early stopping at epoch {epoch}")
        break

ieee_model.load_state_dict(best_state)
print(f"\nBest val AUROC: {best_val_auc:.4f}")

Epoch   1 | loss 0.7267 | val AUROC 0.5343
Epoch  10 | loss 0.6872 | val AUROC 0.6220
Epoch  20 | loss 0.6715 | val AUROC 0.6393
Epoch  30 | loss 0.6600 | val AUROC 0.6454
Epoch  40 | loss 0.6525 | val AUROC 0.6645
Epoch  50 | loss 0.6481 | val AUROC 0.6948
Epoch  60 | loss 0.6407 | val AUROC 0.6997
Epoch  70 | loss 0.6343 | val AUROC 0.7048
Epoch  80 | loss 0.6260 | val AUROC 0.7355
Epoch  90 | loss 0.6171 | val AUROC 0.7509
Epoch 100 | loss 0.6052 | val AUROC 0.7552

Best val AUROC: 0.7604


In [21]:
# ── IEEE-CIS: run through the same execute_streaming_pipeline as everything
# else — no changes to that function needed. has_account_structure=True
# since card1 gives Gate 1 genuine recurring history to work with.

results["ieeecis"] = execute_streaming_pipeline(
    "ieeecis", ieeecis_data, ieee_model,
    has_account_structure=True,
    max_test_samples=(None if device.type == "cuda" else scfg.MAX_TEST_SAMPLES),
)

print(f"\n--- IEEECIS Dual-Gate System Performance vs Baseline (same val-tuned threshold for both) ---")
baseline_metrics = evaluate_predictions(
    results["ieeecis"]["y_true_val"], results["ieeecis"]["y_pred_val"],
    results["ieeecis"]["y_true_test"], results["ieeecis"]["y_pred_baseline"], name="ieeecis-Baseline")
dual_gate_metrics = evaluate_predictions(
    results["ieeecis"]["y_true_val"], results["ieeecis"]["y_pred_val"],
    results["ieeecis"]["y_true_test"], results["ieeecis"]["y_pred_dual_gate"], name="ieeecis-DualGate")
print()


[STREAMING IEEECIS] Precomputing base embeddings & adjacency...
[ieeecis] fraction of test nodes with zero in-neighbors: 0.00%
[diagnostic] raw connected components before capping: 41672
[diagnostic] oversized components: 140 chunked via BFS (topologically local), 0 chunked via sorted-index fallback (BFS/connected_components disagreed)
[community structure] 41956 communities (capped at 500) | largest holds 0.08% of nodes | median size 3

--- IEEECIS Community-Welford z-score by true label (test sample, diagnostic only) ---
Fraud (n=213): mean=0.9664 | median=0.3887 | p95=1.3060
Legit (n=228): mean=0.5053 | median=0.3886 | p95=1.2279
[ieeecis] Community-Welford verdict: NO MEANINGFUL SEPARATION (effect size 0.16)
Calibrating thresholds from validation distribution...
Calibrated TAU_GLOBAL: 0.4149 | TAU_LOCAL: 0.0129
Running the FULL test period: 88581 transactions.
[ieeecis] test-window fraud prevalence: 3.4804% (3083 / 88581)


Processing ieeecis Stream:   0%|          | 0/88581 [00:00<?, ?it/s]


--- IEEECIS Gate Trigger & Purity Diagnostics ---
GATE 1 — Account Welford Tripwire  [SCORES dg_score]
  Fired: 1572 / 88581
  Precision: 0.0254  (40 fraud, 1532 legit)

Distance-Anomaly Check (global/local)  [non-scoring — candidate fallback signal]
  Fired: 9615 / 88581
  Precision: 0.0502  (483 fraud, 9132 legit)

GATE 2 — Community-Welford Co-Activation Check  [tiered confirmation — SCORES dg_score via confidence tier]
  Community-Welford verdict (unconditioned diagnostic sample, see above): NO MEANINGFUL SEPARATION (effect size 0.16)
  Community-Welford verdict (population Gate 1 NEVER flagged, n_fraud=300, n_legit=300): NO INDEPENDENT SEPARATION (effect size 0.22)
  Gate 1 alone (unchanged): 1572 fired, precision 0.0254  (40 fraud, 1532 legit)
  ── split by confidence tier (CAUTION: computed only on the Gate-1-flagged
     population — see 'never flagged' verdict above for the unconditioned test) ──
  HIGH   (Gate 1 + Gate 2 confirmed): 1140 / 1572
    Precision: 0.0351  (40 fra

In [22]:
test_entity_sizes_full_history = ieee_raw.groupby("entity_key").size()
test_entities = ieee_test["entity_key"]
test_txn_burned_in = test_entities.map(test_entity_sizes_full_history).fillna(0) >= scfg.BURN_IN_MIN_NEIGHBORS
print(f"Test transactions from burned-in entities: {test_txn_burned_in.sum():,} / {len(ieee_test):,} "
      f"({test_txn_burned_in.mean()*100:.1f}%)")

print("Community Welford z-threshold:", scfg.COMMUNITY_WELFORD_Z_THRESH)

Test transactions from burned-in entities: 81,966 / 88,581 (92.5%)
Community Welford z-threshold: 3.0


In [23]:
for df in (ieee_train, ieee_val, ieee_test, ieee_raw):
    df["entity_key"] = (
        df["card1"].astype(str) + "_" +
        df["card2"].fillna(-1).astype(str) + "_" +
        df["addr1"].fillna(-1).astype(str)
    )

all_accounts = pd.concat([ieee_train[["entity_key"]], ieee_val[["entity_key"]], ieee_test[["entity_key"]]]).drop_duplicates()
IEEE_ACCT_TO_IDX = {acc: i for i, acc in enumerate(all_accounts["entity_key"].values)}
N_ACCOUNTS = len(IEEE_ACCT_TO_IDX)

print(f"Entities: {N_ACCOUNTS:,}")
print(f"Median entity size: {ieee_raw.groupby('entity_key').size().median():.0f}")
print(f"Mean entity size: {ieee_raw.groupby('entity_key').size().mean():.1f}")
print(f"Entities with >=2 transactions: {(ieee_raw.groupby('entity_key').size() >= 2).sum():,}")

Entities: 41,672
Median entity size: 2
Mean entity size: 14.2
Entities with >=2 transactions: 24,860


In [24]:
# ── Composite entity key instead of card1 alone ──────────────────────────────
# card1's max entity size (~14,932 transactions) indicates it's a coarse,
# collision-prone bucket, not a true 1:1 card fingerprint -- multiple real
# cards very likely hash into the same card1 value. This is the standard
# fix used in published IEEE-CIS approaches: a composite key drastically
# reduces false merging of distinct cards into one entity.

for df in (ieee_train, ieee_val, ieee_test, ieee_raw):
    df["entity_key"] = (
        df["card1"].astype(str) + "_" +
        df["card2"].fillna(-1).astype(str) + "_" +
        df["addr1"].fillna(-1).astype(str) + "_" +
        df["D1"].fillna(-1).astype(str)
    )

all_accounts = pd.concat([ieee_train[["entity_key"]], ieee_val[["entity_key"]], ieee_test[["entity_key"]]]).drop_duplicates()
IEEE_ACCT_TO_IDX = {acc: i for i, acc in enumerate(all_accounts["entity_key"].values)}
N_ACCOUNTS = len(IEEE_ACCT_TO_IDX)

print(f"Entities under composite key: {N_ACCOUNTS:,}  (vs {ieee_raw['card1'].nunique():,} under card1 alone)")
print(f"Median entity size: {ieee_raw.groupby('entity_key').size().median():.0f}  "
      f"(vs {ieee_raw.groupby('card1').size().median():.0f} under card1 alone)")


Entities under composite key: 248,038  (vs 13,553 under card1 alone)
Median entity size: 1  (vs 4 under card1 alone)


In [25]:
print("Burn-in threshold:", scfg.BURN_IN_MIN_NEIGHBORS)
entity_sizes = ieee_raw.groupby("entity_key").size()
print(f"Entities clearing burn-in: {(entity_sizes >= scfg.BURN_IN_MIN_NEIGHBORS).sum():,} "
      f"({(entity_sizes >= scfg.BURN_IN_MIN_NEIGHBORS).mean()*100:.1f}%)")

Burn-in threshold: 5
Entities clearing burn-in: 13,845 (5.6%)


In [26]:
# Extract AMLSim results
aml_out = results["amlsim"]

gate1_fp_rate = aml_out["gate1_fp"] / max(1, aml_out["gate1_triggers"])
dual_gate_fp_rate = aml_out["gate2_fp"] / max(1, aml_out["gate2_triggers"])

print(f"Gate 1 Only FP Rate: {gate1_fp_rate:.2%}")
print(f"Dual-Gate FP Rate: {dual_gate_fp_rate:.2%}")

Gate 1 Only FP Rate: 0.79%
Dual-Gate FP Rate: 0.00%


In [28]:
ablation_results = {}

for l_hops in [1, 2]:
    print(f"\nRunning Ablation for L_HOPS = {l_hops}...")
    scfg.L_HOPS = l_hops
    
    # ADDED: max_test_samples=None to force the full 266k evaluation
    out = execute_streaming_pipeline(
        "amlsim", 
        assets["amlsim"]["data"], 
        assets["amlsim"]["model"], 
        has_account_structure=True,
        max_test_samples=None 
    )
    
    # Evaluate metrics
    metrics = evaluate_predictions(out["y_true_val"], out["y_pred_val"], out["y_true_test"], out["y_pred_dual_gate"])
    
    gate1_fp_rate = out["gate1_fp"] / max(1, out["gate1_triggers"])
    dual_gate_fp_rate = out["gate2_fp"] / max(1, out["gate2_triggers"])
    
    ablation_results[f"L={l_hops}"] = {
        "AUROC": metrics["auroc"],
        "Gate1_FP": gate1_fp_rate,
        "DualGate_FP": dual_gate_fp_rate
    }

print("\n--- Corrected Ablation Results ---")
for k, v in ablation_results.items():
    print(f"{k}: AUROC={v['AUROC']:.4f} | Gate1_FP={v['Gate1_FP']:.2%} | DualGate_FP={v['DualGate_FP']:.2%}")


Running Ablation for L_HOPS = 1...

[STREAMING AMLSIM] Precomputing base embeddings & adjacency...
[amlsim] fraction of test nodes with zero in-neighbors: 0.00%
[diagnostic] raw connected components before capping: 1
[diagnostic] oversized components: 1 chunked via BFS (topologically local), 0 chunked via sorted-index fallback (BFS/connected_components disagreed)
[community structure] 2647 communities (capped at 500) | largest holds 0.04% of nodes | median size 500

--- AMLSIM Community-Welford z-score by true label (test sample, diagnostic only) ---
Fraud (n=300): mean=16.9881 | median=1.6358 | p95=3.8884
Legit (n=300): mean=0.7964 | median=0.6383 | p95=1.9098
[amlsim] Community-Welford verdict: NO MEANINGFUL SEPARATION (effect size 0.13)
Calibrating thresholds from validation distribution...
Calibrated TAU_GLOBAL: 0.5858 | TAU_LOCAL: 0.0000
Running the FULL test period: 266199 transactions.
[amlsim] test-window fraud prevalence: 0.1416% (377 / 266199)


Processing amlsim Stream:   0%|          | 0/266199 [00:00<?, ?it/s]


--- AMLSIM Gate Trigger & Purity Diagnostics ---
GATE 1 — Account Welford Tripwire  [SCORES dg_score]
  Fired: 380 / 266199
  Precision: 0.9921  (377 fraud, 3 legit)

Distance-Anomaly Check (global/local)  [non-scoring — candidate fallback signal]
  Fired: 11958 / 266199
  Precision: 0.0018  (22 fraud, 11936 legit)

GATE 2 — Community-Welford Co-Activation Check  [tiered confirmation — SCORES dg_score via confidence tier]
  Community-Welford verdict (unconditioned diagnostic sample, see above): NO MEANINGFUL SEPARATION (effect size 0.13)
  Community-Welford verdict (population Gate 1 NEVER flagged, n_fraud=0, n_legit=300): INSUFFICIENT SAMPLE — too few burned-in nodes outside Gate 1's catches
  Gate 1 alone (unchanged): 380 fired, precision 0.9921  (377 fraud, 3 legit)
  ── split by confidence tier (CAUTION: computed only on the Gate-1-flagged
     population — see 'never flagged' verdict above for the unconditioned test) ──
  HIGH   (Gate 1 + Gate 2 confirmed): 33 / 380
    Precision

Processing amlsim Stream:   0%|          | 0/266199 [00:00<?, ?it/s]


--- AMLSIM Gate Trigger & Purity Diagnostics ---
GATE 1 — Account Welford Tripwire  [SCORES dg_score]
  Fired: 380 / 266199
  Precision: 0.9921  (377 fraud, 3 legit)

Distance-Anomaly Check (global/local)  [non-scoring — candidate fallback signal]
  Fired: 23894 / 266199
  Precision: 0.0022  (52 fraud, 23842 legit)

GATE 2 — Community-Welford Co-Activation Check  [tiered confirmation — SCORES dg_score via confidence tier]
  Community-Welford verdict (unconditioned diagnostic sample, see above): NO MEANINGFUL SEPARATION (effect size 0.13)
  Community-Welford verdict (population Gate 1 NEVER flagged, n_fraud=0, n_legit=300): INSUFFICIENT SAMPLE — too few burned-in nodes outside Gate 1's catches
  Gate 1 alone (unchanged): 380 fired, precision 0.9921  (377 fraud, 3 legit)
  ── split by confidence tier (CAUTION: computed only on the Gate-1-flagged
     population — see 'never flagged' verdict above for the unconditioned test) ──
  HIGH   (Gate 1 + Gate 2 confirmed): 33 / 380
    Precision